In [1]:
import pathlib

import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd

import jax
import jax.numpy as jnp
import jax.random as jr
import optax
import equinox as eqx
from jaxtyping import Array, Float, Int, PyTree, PRNGKeyArray

import torch
from torch.utils.data import DataLoader, Subset

import altair as alt

import trapallada.data
import trapallada.model.signal

So that imported modules are automatically reloaded

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Directory in which the data lives

In [3]:
input_directory = pathlib.Path.cwd().parent / "data"
assert input_directory.exists()

`base_filename` identifies a particular dataset

In [4]:
# base_filename = 'small'
base_filename = '4_classes_modulated'

signals_file = input_directory / f'{base_filename}_signals.pt'
assert signals_file.exists()

metadata_file = input_directory / f'{base_filename}_labels.parquet'
assert metadata_file.exists

In [6]:
ds = trapallada.data.SignalsDataset(signals_file, metadata_file)
len(ds)

12983

In [ ]:
ds.metadata_df.sample(n=10)

,label,SNR,color,file name
3152,FM,1.000,None,FM/FM_SNR1_Simul381.mat
11484,carrier,3.162,None,singlePeriod/carrier_SNR3.162_Simul77.mat
2318,FM,10.000,None,FM/FM_SNR10_Simul940.mat
5386,AM,31.620,None,AM/AM_SNR31.62_Simul979.mat
12944,noise,NaN,1.2982,noise/noise_color1.2982.mat
9911,carrier,31.620,None,singlePeriod/carrier_SNR31.62_Simul784.mat
9175,carrier,1.000,None,singlePeriod/carrier_SNR1_Simul315.mat
4641,AM,31.620,None,AM/AM_SNR31.62_Simul848.mat
11665,carrier,10.000,None,singlePeriod/carrier_SNR10_Simul81.mat
7581,AM,3.162,None,AM/AM_SNR3.162_Simul117.mat


In [ ]:
ds.signals.shape

torch.Size([12983, 1, 10000])

In [ ]:
i_signal_max = torch.where(ds.signals == ds.signals.max())[0].item()
ds.metadata_df.iloc[i_signal_max]

label                              noise
SNR                                  NaN
color                             1.9945
file name    noise/noise_color1.9945.mat
Name: 12795, dtype: object

In [ ]:
n_rows = 2000

In [ ]:
y = ds[i_signal_max][1]
df = pd.DataFrame(y.squeeze(), columns=['signal'])
alt.Chart(df.iloc[:n_rows].reset_index()).mark_line().encode(
    x='index', y='signal'
)


alt.Chart(...)

In [ ]:
i_noise = ds.metadata_df['label'] == 'noise'
i_noise
ds.metadata_df[i_noise]

,label,SNR,color,file name
12000,noise,NaN,0.43625,noise/noise_color0.43625.mat
12001,noise,NaN,1.9453,noise/noise_color1.9453.mat
12002,noise,NaN,0.93627,noise/noise_color0.93627.mat
12003,noise,NaN,1.3204,noise/noise_color1.3204.mat
12004,noise,NaN,0.19382,noise/noise_color0.19382.mat
...,...,...,...,...
12978,noise,NaN,1.6987,noise/noise_color1.6987.mat
12979,noise,NaN,1.9692,noise/noise_color1.9692.mat
12980,noise,NaN,0.19187,noise/noise_color0.19187.mat
12981,noise,NaN,1.3106,noise/noise_color1.3106.mat


Data for *only-noise* signals

In [ ]:
noise_signals = ds.signals[i_noise].squeeze().T
noise_metadata_df = ds.metadata_df[i_noise]
noise_signals_df = pd.DataFrame(noise_signals, columns=noise_metadata_df['file name'].str.extract(r'.*/(.*).(?:jpg|mat)')[0])
noise_signals_df

,noise_color0.43625,noise_color1.9453,noise_color0.93627,noise_color1.3204,noise_color0.19382,noise_color0.64144,noise_color1.8508,noise_color1.5291,noise_color1.1959,noise_color1.6255,...,noise_color1.0827,noise_color1.7431,noise_color0.74522,noise_color1.3849,noise_color1.2807,noise_color1.6987,noise_color1.9692,noise_color0.19187,noise_color1.3106,noise_color0.71187
0,-0.245115,-0.096280,-0.678100,1.782245,0.153667,-0.812335,-0.615126,-0.342772,-0.105996,1.157561,...,0.105056,0.830267,0.253802,0.261214,-1.008204,0.049363,0.501141,1.001249,-1.002293,0.525035
1,0.775136,0.253847,-0.139147,-0.651941,-0.743991,-1.631504,-0.175083,-1.401462,0.072311,-0.481155,...,-1.593484,0.555988,0.003520,1.166614,-0.073747,0.964568,0.747118,-0.343929,0.125302,1.908232
2,-0.195425,0.848270,0.021620,-0.743316,0.643989,0.426455,0.082786,-1.242709,-1.558279,-0.761694,...,-1.039785,-0.957565,-0.432666,1.285646,-0.656491,0.265001,-0.072051,-0.111900,-1.788730,1.098732
3,0.170898,0.558168,-1.547086,0.042588,-0.246362,0.034864,0.055612,-1.230603,-1.428367,-0.223430,...,-2.504379,-2.067653,0.619696,0.191794,-0.347355,0.427403,-0.058699,1.250695,-1.096658,-0.297516
4,0.397939,-0.716962,0.680718,0.533967,0.276412,2.508490,0.491531,-0.958019,-1.812329,-0.195751,...,-1.449681,-1.397527,0.753677,-0.803490,-0.607208,1.445568,-0.085994,1.662358,-3.723509,-0.793372
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,-0.036640,-45.173820,-0.024726,-5.972010,-0.827487,-0.730839,3.117975,-3.813413,1.979163,-5.931253,...,0.943191,3.690809,-0.182349,-2.010277,1.744213,-3.289083,21.620185,0.401380,3.738484,-2.698845
9996,0.792380,-45.525483,-0.870316,-6.108037,-1.480484,1.148227,3.747931,-2.615573,2.743834,-7.266403,...,1.175099,1.812873,-0.007134,-2.593531,1.889870,-3.458971,21.407262,0.392726,3.481368,-1.718749
9997,-0.014244,-46.229013,-1.148884,-4.862473,-0.334671,-0.535666,1.456177,-1.714417,1.428454,-5.585585,...,1.841487,1.921700,1.225760,-1.087202,1.701638,-2.364747,21.969806,-0.442795,2.271954,-1.825558
9998,1.913293,-46.630912,-0.967016,-5.109375,0.057115,0.512249,2.647690,-1.471321,1.591795,-8.531293,...,0.042387,0.727335,1.245694,-2.476316,2.043983,-1.721829,21.740915,1.188674,2.049165,-1.461592


In [ ]:
some_noise_signals_df = noise_signals_df.sample(n=50, axis=1)
some_noise_signals_df.columns = some_noise_signals_df.columns.str.replace('.', '')
alt.Chart(
    some_noise_signals_df.iloc[:n_rows].reset_index()
).transform_fold(
    some_noise_signals_df.columns.to_list()
).mark_line().encode(
    x='index:Q', y='value:Q', color='key:N'
).properties(width=1_200, height=400)

alt.Chart(...)